# Unidade III — Pré-processamento de Dados

## Qualidade, limpeza e integração

**Carga estimada:** 3 horas  
**Pré-requisitos:** pandas, estatística descritiva e tipos de atributos.

> **Pergunta norteadora:** como transformar registros problemáticos em uma base analisável sem apagar evidências nem inventar certezas?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- diagnosticar completude, validade, consistência, unicidade e atualidade;
- tratar ausências, duplicatas e valores inválidos com regras justificadas;
- integrar tabelas controlando cardinalidade e correspondência de entidades;
- comparar indicadores antes e depois sem modificar os dados brutos.


In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Qualidade depende do uso

Qualidade não significa perfeição abstrata. Um dado é adequado quando sustenta a finalidade declarada. **Completude** examina ausências; **validade**, conformidade com domínios e formatos; **consistência**, compatibilidade entre campos e fontes; **unicidade**, registros repetidos; e **atualidade**, adequação temporal. Acurácia, isto é, proximidade do valor real, frequentemente exige uma fonte externa confiável.

O exemplo abaixo é sintético e introduz problemas conhecidos para que possamos verificar o efeito de cada decisão. A cópia `clientes_brutos` será preservada; toda alteração ocorrerá em outra tabela.


In [2]:
DATA_REFERENCIA = pd.Timestamp("2026-01-01")
LIMITE_ATUALIZACAO = DATA_REFERENCIA - pd.DateOffset(years=2)

clientes_brutos = pd.DataFrame({
    "cliente_id": [101, 102, 103, 104, 104, 105, 106, 107],
    "idade": [34, np.nan, 29, 150, 150, 45, 38, 52],
    "cidade": ["Recife", " recife " , "Olinda", "Recife", "Recife", "Paulista", "OLINDA", "Recife"],
    "uf": ["PE", "PE", "PE", "PE", "PE", "PE", "PB", "PE"],
    "email": ["ana@exemplo.br", "BRUNO@EXEMPLO.BR", None, "dora@exemplo.br", "dora@exemplo.br", "eva@exemplo.br", "f@exemplo.br", "g@exemplo.br"],
    "mensalidade": [89.9, 110.0, np.nan, 95.0, 95.0, -20.0, 130.0, 120.0],
    "data_atualizacao": pd.to_datetime([
        "2025-11-15", "2025-10-02", "2025-09-10", "2025-08-01",
        "2025-08-01", "2025-06-20", "2025-04-12", "2020-01-10",
    ]),
})
clientes_brutos_original = clientes_brutos.copy(deep=True)

clientes_brutos


,cliente_id,idade,cidade,uf,email,mensalidade,data_atualizacao
0,101,34.0,Recife,PE,ana@exemplo.br,89.9,2025-11-15
1,102,NaN,recife,PE,BRUNO@EXEMPLO.BR,110.0,2025-10-02
2,103,29.0,Olinda,PE,None,NaN,2025-09-10
3,104,150.0,Recife,PE,dora@exemplo.br,95.0,2025-08-01
4,104,150.0,Recife,PE,dora@exemplo.br,95.0,2025-08-01
5,105,45.0,Paulista,PE,eva@exemplo.br,-20.0,2025-06-20
6,106,38.0,OLINDA,PB,f@exemplo.br,130.0,2025-04-12
7,107,52.0,Recife,PE,g@exemplo.br,120.0,2020-01-10


### Como reconhecer cada dimensão nos dados

A tabela de clientes acima contém pelo menos um exemplo explícito de cada dimensão:

| Dimensão | Exemplo na tabela | Por que é um problema? |
|---|---|---|
| **Completude** | cliente 102: idade ausente | Um valor necessário não foi registrado. |
| **Validade** | cliente 104: idade 150 | Viola o domínio declarado de 18 a 100 anos. |
| **Consistência** | cliente 106: cidade Olinda e UF PB | A cidade informada é incompatível com a unidade federativa. |
| **Unicidade** | cliente 104 aparece em duas linhas | A mesma entidade possui mais de uma representação. |
| **Atualidade** | cliente 107: atualização em 2020-01-10 | Na referência 2026-01-01, excede o limite declarado de dois anos. |

Há exemplos adicionais: o cliente 103 não possui e-mail nem mensalidade, e o cliente 105 possui mensalidade negativa. Uma linha pode violar mais de uma dimensão, e uma dimensão pode aparecer em várias linhas.

## Diagnóstico antes da correção

O relatório torna os critérios auditáveis. Ausência é medida por coluna; duplicidade usa a chave de entidade; validade requer regras de domínio; consistência compara cidade e UF; atualidade depende de uma data de referência. Aqui, idades devem estar entre 18 e 100 anos, mensalidades devem ser positivas, Recife, Olinda e Paulista devem estar em PE, e registros anteriores a 2024-01-01 são considerados desatualizados na data de referência 2026-01-01. Essas regras pertencem ao estudo de caso e não devem ser transferidas para outra base sem validação.


In [3]:
CIDADE_PARA_UF = {"Recife": "PE", "Olinda": "PE", "Paulista": "PE"}

def relatorio_qualidade(df: pd.DataFrame) -> pd.Series:
    """Resume problemas definidos para este estudo de caso."""
    cidade_normalizada = df["cidade"].str.strip().str.title()
    uf_esperada = cidade_normalizada.map(CIDADE_PARA_UF)
    cidade_uf_inconsistente = uf_esperada.notna() & df["uf"].ne(uf_esperada)
    data_atualizacao = pd.to_datetime(df["data_atualizacao"])
    return pd.Series({
        "linhas": len(df),
        "celulas_ausentes": int(df.isna().sum().sum()),
        "ids_duplicados": int(df.duplicated("cliente_id", keep=False).sum()),
        "idades_invalidas": int((~df["idade"].between(18, 100) & df["idade"].notna()).sum()),
        "mensalidades_invalidas": int((df["mensalidade"] <= 0).sum()),
        "cidade_uf_inconsistentes": int(cidade_uf_inconsistente.sum()),
        "registros_desatualizados": int((data_atualizacao < LIMITE_ATUALIZACAO).sum()),
    })

antes = relatorio_qualidade(clientes_brutos)
assert antes.to_dict() == {
    "linhas": 8,
    "celulas_ausentes": 3,
    "ids_duplicados": 2,
    "idades_invalidas": 2,
    "mensalidades_invalidas": 1,
    "cidade_uf_inconsistentes": 1,
    "registros_desatualizados": 1,
}
antes.to_frame("antes")


,antes
linhas,8
celulas_ausentes,3
ids_duplicados,2
idades_invalidas,2
mensalidades_invalidas,1
cidade_uf_inconsistentes,1
registros_desatualizados,1


## Limpeza rastreável

A ordem importa: primeiro padronizamos texto; depois harmonizamos cidade e UF usando a referência declarada; marcamos valores impossíveis como ausentes; consolidamos duplicatas de entidade; e, por fim, imputamos medianas. Imputação reduz ausências, mas não recupera o valor verdadeiro e pode reduzir artificialmente a variabilidade. Em modelagem supervisionada, seus parâmetros devem ser aprendidos apenas no treino, como veremos no próximo notebook.

Atualidade exige tratamento diferente: uma data antiga não informa qual seria o valor atual. Por isso, o registro do cliente 107 permanecerá sinalizado para recadastramento, em vez de receber uma data inventada.


In [4]:
clientes_limpos = clientes_brutos.copy(deep=True)
clientes_limpos["cidade"] = clientes_limpos["cidade"].str.strip().str.title()
clientes_limpos["email"] = clientes_limpos["email"].str.strip().str.lower()
uf_referenciada = clientes_limpos["cidade"].map(CIDADE_PARA_UF)
clientes_limpos["uf"] = uf_referenciada.fillna(clientes_limpos["uf"])
clientes_limpos.loc[~clientes_limpos["idade"].between(18, 100), "idade"] = np.nan
clientes_limpos.loc[clientes_limpos["mensalidade"] <= 0, "mensalidade"] = np.nan
clientes_limpos = clientes_limpos.drop_duplicates("cliente_id", keep="first")
for coluna in ["idade", "mensalidade"]:
    clientes_limpos[coluna] = clientes_limpos[coluna].fillna(clientes_limpos[coluna].median())

pd.testing.assert_frame_equal(clientes_brutos, clientes_brutos_original)
clientes_limpos


,cliente_id,idade,cidade,uf,email,mensalidade,data_atualizacao
0,101,34.0,Recife,PE,ana@exemplo.br,89.9,2025-11-15
1,102,38.0,Recife,PE,bruno@exemplo.br,110.0,2025-10-02
2,103,29.0,Olinda,PE,None,110.0,2025-09-10
3,104,38.0,Recife,PE,dora@exemplo.br,95.0,2025-08-01
5,105,45.0,Paulista,PE,eva@exemplo.br,110.0,2025-06-20
6,106,38.0,Olinda,PE,f@exemplo.br,130.0,2025-04-12
7,107,52.0,Recife,PE,g@exemplo.br,120.0,2020-01-10


## Integração e cardinalidade

Integração não é apenas chamar `merge`. Precisamos definir a entidade, harmonizar chaves e declarar a cardinalidade esperada. A opção `validate="one_to_one"` faz a operação falhar se qualquer tabela tiver mais de uma linha por cliente. Em dados reais, nomes ou e-mails aproximados exigem resolução de entidades, revisão de falsos pares e proteção de dados pessoais.


In [5]:
contratos = pd.DataFrame({
    "cliente_id": [101, 102, 103, 104, 105, 106, 108],
    "plano": ["Básico", "Pro", "Básico", "Pro", "Básico", "Pro", "Básico"],
})

integrados = clientes_limpos.merge(
    contratos, on="cliente_id", how="left", validate="one_to_one", indicator=True
)
assert len(integrados) == len(clientes_limpos) == 7
assert integrados["_merge"].eq("left_only").sum() == 1
integrados[["cliente_id", "plano", "_merge"]]


,cliente_id,plano,_merge
0,101,Básico,both
1,102,Pro,both
2,103,Básico,both
3,104,Pro,both
4,105,Básico,both
5,106,Pro,both
6,107,NaN,left_only


In [6]:
depois = relatorio_qualidade(clientes_limpos)
assert depois.to_dict() == {
    "linhas": 7,
    "celulas_ausentes": 1,
    "ids_duplicados": 0,
    "idades_invalidas": 0,
    "mensalidades_invalidas": 0,
    "cidade_uf_inconsistentes": 0,
    "registros_desatualizados": 1,
}
comparacao = pd.concat([antes.rename("antes"), depois.rename("depois")], axis=1)
comparacao["variacao"] = comparacao["depois"] - comparacao["antes"]
comparacao


,antes,depois,variacao
linhas,8,7,-1
celulas_ausentes,3,1,-2
ids_duplicados,2,0,-2
idades_invalidas,2,0,-2
mensalidades_invalidas,1,0,-1
cidade_uf_inconsistentes,1,0,-1
registros_desatualizados,1,1,0


O relatório mostra que duplicidade, valores inválidos e inconsistência cidade–UF foram eliminados segundo regras declaradas. Uma ausência permanece no e-mail porque não existe base defensável para inventar esse identificador. O registro desatualizado também permanece sinalizado: corrigir atualidade requer obter informação nova do cliente, não trocar a data automaticamente.

A redução dos indicadores não prova acurácia. A idade imputada tornou-se completa e válida, mas continua sendo uma estimativa; valores plausíveis também podem estar factualmente errados. A junção revela clientes sem contrato correspondente, que devem ser investigados e não removidos silenciosamente.

> **U03-NB01-V01 — Verifique seu entendimento:** por que substituir uma idade impossível pela mediana não torna esse valor conhecido nem garante acurácia?

> **U03-NB01-E01 — Exercício:** adapte `relatorio_qualidade` para incluir a proporção de ausências por coluna e o número de clientes sem contrato. Entregue a função, a tabela resultante e duas interpretações.


## Síntese

- Completude, validade, consistência, unicidade e atualidade detectam problemas diferentes.
- Qualidade é definida em relação ao uso, ao domínio e a uma referência temporal explícita.
- Dados brutos devem permanecer imutáveis; transformações precisam ser rastreáveis.
- Imputação expressa uma decisão sob incerteza.
- Junções exigem chave, cardinalidade e auditoria de correspondências.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 2, seção 2.4.
- PANDAS DEVELOPMENT TEAM. *pandas documentation*: missing data e merge. Versão 2.x.
